# 1.Word Count Program

In [1]:
!apt-get update -qq
!apt-get install openjdk-8-jdk-headless -qq

!wget -q https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz
!mv hadoop-3.3.6 hadoop

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/content/hadoop"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["HADOOP_HOME"] + "/bin:" + os.environ["PATH"]

!java -version
!javac -version

openjdk version "1.8.0_482"
OpenJDK Runtime Environment (build 1.8.0_482-8u482-ga~us1-0ubuntu1~22.04-b08)
OpenJDK 64-Bit Server VM (build 25.482-b08, mixed mode)
javac 1.8.0_482


In [3]:
%%bash
cat > input.txt << EOF
hadoop big data hadoop
big data analytics
hadoop analytics
EOF

In [4]:
%%bash
cat > WordCount.java << 'EOF'
import java.io.IOException;
import java.util.StringTokenizer;
import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.*;
import org.apache.hadoop.mapreduce.*;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;

public class WordCount {

    public static class TokenizerMapper
        extends Mapper<Object, Text, Text, IntWritable>{

        private final static IntWritable one = new IntWritable(1);
        private Text word = new Text();

        public void map(Object key, Text value, Context context)
                throws IOException, InterruptedException {

            StringTokenizer itr = new StringTokenizer(value.toString());
            while (itr.hasMoreTokens()) {
                word.set(itr.nextToken());
                context.write(word, one);
            }
        }
    }

    public static class IntSumReducer
        extends Reducer<Text, IntWritable, Text, IntWritable>{

        public void reduce(Text key, Iterable<IntWritable> values,
                           Context context)
                throws IOException, InterruptedException {

            int sum = 0;
            for (IntWritable val : values) {
                sum += val.get();
            }
            context.write(key, new IntWritable(sum));
        }
    }

    public static void main(String[] args) throws Exception {

        Configuration conf = new Configuration();
        Job job = Job.getInstance(conf, "word count");

        job.setJarByClass(WordCount.class);
        job.setMapperClass(TokenizerMapper.class);

        // ✅ Combiner
        job.setCombinerClass(IntSumReducer.class);

        job.setReducerClass(IntSumReducer.class);

        job.setOutputKeyClass(Text.class);
        job.setOutputValueClass(IntWritable.class);

        FileInputFormat.addInputPath(job, new Path(args[0]));
        FileOutputFormat.setOutputPath(job, new Path(args[1]));

        System.exit(job.waitForCompletion(true) ? 0 : 1);
    }
}
EOF

In [5]:
!javac --release 8 -classpath `hadoop classpath` WordCount.java
!jar cf wc.jar WordCount*.class

javac: invalid flag: --release
Usage: javac <options> <source files>
use -help for a list of possible options


In [6]:
!update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java
!update-alternatives --set javac /usr/lib/jvm/java-8-openjdk-amd64/bin/javac

update-alternatives: using /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java to provide /usr/bin/java (java) in manual mode
update-alternatives: using /usr/lib/jvm/java-8-openjdk-amd64/bin/javac to provide /usr/bin/javac (javac) in manual mode


In [7]:
!java -version
!javac -version

openjdk version "1.8.0_482"
OpenJDK Runtime Environment (build 1.8.0_482-8u482-ga~us1-0ubuntu1~22.04-b08)
OpenJDK 64-Bit Server VM (build 25.482-b08, mixed mode)
javac 1.8.0_482


In [8]:
!javac -source 1.8 -target 1.8 -classpath `hadoop classpath` WordCount.java
!jar cf wc.jar WordCount*.class
!rm -rf output_wc_java
!hadoop jar wc.jar WordCount input.txt output_wc_java

2026-03-30 15:47:36,262 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-03-30 15:47:36,517 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-03-30 15:47:36,517 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-03-30 15:47:36,805 WARN mapreduce.JobResourceUploader: Hadoop command-line option parsing not performed. Implement the Tool interface and execute your application with ToolRunner to remedy this.
2026-03-30 15:47:37,155 INFO input.FileInputFormat: Total input files to process : 1
2026-03-30 15:47:37,220 INFO mapreduce.JobSubmitter: number of splits:1
2026-03-30 15:47:37,667 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1588589183_0001
2026-03-30 15:47:37,667 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-03-30 15:47:38,034 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-03-30 15:47:38,035 INFO mapreduce.Job: Running job: job_local1588589183_

In [10]:
!cat output_wc_java/part-r-00000

analytics	2
big	2
data	2
hadoop	3


# 2.KeyWordSearch Count

In [11]:
%%bash
cat > input.txt << EOF
hadoop big data hadoop
big data analytics
hadoop analytics
EOF

In [12]:
%%bash
cat > KeywordSearch.java << 'EOF'
import java.io.IOException;
import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.*;
import org.apache.hadoop.mapreduce.*;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;

public class KeywordSearch {

    public static class SearchMapper
        extends Mapper<Object, Text, Text, Text>{

        private Text result = new Text();
        private String keyword = "hadoop";

        public void map(Object key, Text value, Context context)
                throws IOException, InterruptedException {

            String line = value.toString();

            if (line.toLowerCase().contains(keyword)) {
                result.set(line);
                context.write(new Text("Found"), result);
            }
        }
    }

    public static class IdentityReducer
        extends Reducer<Text, Text, Text, Text>{

        public void reduce(Text key, Iterable<Text> values,
                           Context context)
                throws IOException, InterruptedException {

            for (Text val : values) {
                context.write(key, val);
            }
        }
    }

    public static void main(String[] args) throws Exception {

        Configuration conf = new Configuration();
        Job job = Job.getInstance(conf, "keyword search");

        job.setJarByClass(KeywordSearch.class);
        job.setMapperClass(SearchMapper.class);
        job.setReducerClass(IdentityReducer.class);

        job.setOutputKeyClass(Text.class);
        job.setOutputValueClass(Text.class);

        FileInputFormat.addInputPath(job, new Path(args[0]));
        FileOutputFormat.setOutputPath(job, new Path(args[1]));

        System.exit(job.waitForCompletion(true) ? 0 : 1);
    }
}
EOF

In [13]:
!javac -source 1.8 -target 1.8 -classpath `hadoop classpath` KeywordSearch.java
!jar cf ks.jar KeywordSearch*.class

In [14]:
!rm -rf output_search_java

!hadoop jar ks.jar KeywordSearch input.txt output_search_java

2026-03-30 15:54:29,811 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-03-30 15:54:29,992 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-03-30 15:54:29,992 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-03-30 15:54:30,223 WARN mapreduce.JobResourceUploader: Hadoop command-line option parsing not performed. Implement the Tool interface and execute your application with ToolRunner to remedy this.
2026-03-30 15:54:30,510 INFO input.FileInputFormat: Total input files to process : 1
2026-03-30 15:54:30,589 INFO mapreduce.JobSubmitter: number of splits:1
2026-03-30 15:54:30,963 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1629124665_0001
2026-03-30 15:54:30,964 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-03-30 15:54:31,386 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-03-30 15:54:31,390 INFO mapreduce.Job: Running job: job_local1629124665_

In [15]:
!cat output_search_java/part-r-00000

Found	hadoop analytics
Found	hadoop big data hadoop


# 3.Sorting By Student Name

In [16]:
%%bash
cat > students.txt << EOF
1 Rahul 85
2 Aman 90
3 Zaid 75
EOF

In [17]:
%%bash
cat > SortByName.java << 'EOF'
import java.io.IOException;
import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.*;
import org.apache.hadoop.mapreduce.*;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;

public class SortByName {

    public static class SortMapper
        extends Mapper<Object, Text, Text, Text>{

        public void map(Object key, Text value, Context context)
                throws IOException, InterruptedException {

            String[] parts = value.toString().split(" ");

            if (parts.length >= 3) {
                // KEY = name (for sorting)
                // VALUE = full record
                context.write(new Text(parts[1]), value);
            }
        }
    }

    public static class SortReducer
        extends Reducer<Text, Text, Text, Text>{

        public void reduce(Text key, Iterable<Text> values,
                           Context context)
                throws IOException, InterruptedException {

            for (Text val : values) {
                context.write(key, val);
            }
        }
    }

    public static void main(String[] args) throws Exception {

        Configuration conf = new Configuration();
        Job job = Job.getInstance(conf, "sort by name");

        job.setJarByClass(SortByName.class);
        job.setMapperClass(SortMapper.class);
        job.setReducerClass(SortReducer.class);

        job.setOutputKeyClass(Text.class);
        job.setOutputValueClass(Text.class);

        FileInputFormat.addInputPath(job, new Path(args[0]));
        FileOutputFormat.setOutputPath(job, new Path(args[1]));

        System.exit(job.waitForCompletion(true) ? 0 : 1);
    }
}
EOF

In [18]:
!javac -source 1.8 -target 1.8 -classpath `hadoop classpath` SortByName.java
!jar cf sort.jar SortByName*.class

In [19]:
!rm -rf output_sort_java

!hadoop jar sort.jar SortByName students.txt output_sort_java

2026-03-30 15:57:00,683 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-03-30 15:57:00,780 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-03-30 15:57:00,780 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-03-30 15:57:00,899 WARN mapreduce.JobResourceUploader: Hadoop command-line option parsing not performed. Implement the Tool interface and execute your application with ToolRunner to remedy this.
2026-03-30 15:57:01,057 INFO input.FileInputFormat: Total input files to process : 1
2026-03-30 15:57:01,089 INFO mapreduce.JobSubmitter: number of splits:1
2026-03-30 15:57:01,356 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1687876403_0001
2026-03-30 15:57:01,356 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-03-30 15:57:01,576 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-03-30 15:57:01,580 INFO mapreduce.Job: Running job: job_local1687876403_

In [20]:
!cat output_sort_java/part-r-00000

Aman	2 Aman 90
Rahul	1 Rahul 85
Zaid	3 Zaid 75
